# Streaming Algorithms: Hands-On Implementation

This notebook accompanies the "Streaming Algorithms: Counting in the Data Firehose" slides.

We'll implement and experiment with:
1. **Misra-Gries** for Heavy Hitters
2. **CVM Algorithm** for Distinct Elements
3. Empirical analysis and visualizations

---

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from typing import List, Set, Dict, Tuple
import time

# Set style for plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: Stream Generators

First, let's create some utility functions to generate different types of streams.

In [ ]:
def generate_uniform_stream(n_distinct: int, length: int) -> List[int]:
    """Generate a stream with uniform distribution over n_distinct elements."""
    return [random.randint(1, n_distinct) for _ in range(length)]


def generate_zipf_stream(n_distinct: int, length: int, alpha: float = 1.5) -> List[int]:
    """Generate a stream following Zipf's law (power-law distribution).
    
    In Zipf's law, the frequency of element i is proportional to 1/i^alpha.
    This models many real-world phenomena: word frequencies, city populations, etc.
    """
    # Compute probabilities
    weights = [1.0 / (i ** alpha) for i in range(1, n_distinct + 1)]
    total = sum(weights)
    probs = [w / total for w in weights]
    
    # Generate stream
    return list(np.random.choice(range(1, n_distinct + 1), size=length, p=probs))


def generate_adversarial_stream(n_distinct: int, length: int) -> List[int]:
    """Generate a stream that's 'hard' for sampling algorithms.
    
    Each distinct element appears exactly once at the beginning,
    then we repeat a single element many times.
    """
    # All distinct elements once
    stream = list(range(1, n_distinct + 1))
    # Pad with repetitions of element 1
    stream.extend([1] * (length - n_distinct))
    return stream


# Test the generators
print("Uniform stream (first 20 elements):")
print(generate_uniform_stream(10, 20))

print("\nZipf stream (first 20 elements):")
zipf = generate_zipf_stream(10, 20)
print(zipf)
print(f"Distribution: {Counter(zipf)}")

## Part 2: Misra-Gries Algorithm for Heavy Hitters

The Misra-Gries algorithm finds elements that appear more than `m/k` times in a stream of length `m`.

In [ ]:
class MisraGries:
    """Misra-Gries algorithm for finding heavy hitters.
    
    Guarantees: Every element with frequency > m/k will be in the output.
    Uses O(k) counters.
    """
    
    def __init__(self, k: int):
        """
        Args:
            k: Number of counters (finds elements appearing > m/k times)
        """
        self.k = k
        self.counters: Dict[int, int] = {}
        self.total_processed = 0
    
    def process(self, element: int) -> None:
        """Process a single element from the stream."""
        self.total_processed += 1
        
        if element in self.counters:
            # Element already tracked, increment
            self.counters[element] += 1
        elif len(self.counters) < self.k - 1:
            # Have space for a new element
            self.counters[element] = 1
        else:
            # No space, decrement all counters
            to_delete = []
            for key in self.counters:
                self.counters[key] -= 1
                if self.counters[key] == 0:
                    to_delete.append(key)
            for key in to_delete:
                del self.counters[key]
    
    def process_stream(self, stream: List[int]) -> None:
        """Process an entire stream."""
        for element in stream:
            self.process(element)
    
    def get_candidates(self) -> Dict[int, int]:
        """Return the current heavy hitter candidates with their estimated counts."""
        return dict(self.counters)
    
    def get_frequency_estimates(self) -> Dict[int, float]:
        """Return estimated frequency fractions."""
        return {k: v / self.total_processed for k, v in self.counters.items()}

In [ ]:
# Test Misra-Gries on a Zipf stream
stream = generate_zipf_stream(20, 10000, alpha=1.5)
true_counts = Counter(stream)

mg = MisraGries(k=5)  # Find elements appearing > 10000/5 = 2000 times
mg.process_stream(stream)

print("True top-10 elements (by frequency):")
for elem, count in true_counts.most_common(10):
    print(f"  Element {elem}: {count} ({100*count/len(stream):.1f}%)")

print(f"\nMisra-Gries candidates (k={mg.k}):")
for elem, est_count in sorted(mg.get_candidates().items(), key=lambda x: -x[1]):
    true_count = true_counts[elem]
    print(f"  Element {elem}: estimated {est_count}, true {true_count}")

# Check guarantee: all elements with freq > m/k should be present
threshold = len(stream) / mg.k
print(f"\nThreshold (m/k): {threshold:.0f}")
heavy_hitters = [e for e, c in true_counts.items() if c > threshold]
print(f"True heavy hitters: {heavy_hitters}")
found = set(mg.get_candidates().keys())
print(f"All found: {set(heavy_hitters).issubset(found)}")

## Part 3: The CVM Algorithm for Distinct Elements

The main attraction! We implement the CVM algorithm from Chakraborty, Vinodchandran, and Meel (ESA 2022).

In [ ]:
class CVMDistinctElements:
    """CVM Algorithm for estimating the number of distinct elements in a stream.
    
    Reference: Chakraborty, Vinodchandran, Meel. 
    "Distinct Elements in Streams: An Algorithm for the (Text) Book." ESA 2022.
    
    The algorithm maintains a buffer X and a probability p such that:
    - Each distinct element seen so far is in X independently with probability p
    - When the buffer gets too full, we subsample by half and halve p
    
    Final estimate: |X| / p
    """
    
    def __init__(self, buffer_size: int):
        """
        Args:
            buffer_size: Maximum number of elements to store.
                         Larger = more accurate, more memory.
        """
        self.buffer_size = buffer_size
        self.buffer: Set[int] = set()
        self.p = 1.0
        self.total_processed = 0
        self.subsample_count = 0
    
    def process(self, element: int) -> None:
        """Process a single element from the stream."""
        self.total_processed += 1
        
        # If element is in buffer, remove it first ("re-roll the dice")
        if element in self.buffer:
            self.buffer.remove(element)
        
        # Add with probability p
        if random.random() < self.p:
            self.buffer.add(element)
        
        # Subsample if buffer is too full
        while len(self.buffer) >= self.buffer_size:
            self._subsample()
    
    def _subsample(self) -> None:
        """Subsample the buffer by half and halve p."""
        self.subsample_count += 1
        new_buffer = set()
        for elem in self.buffer:
            if random.random() < 0.5:
                new_buffer.add(elem)
        self.buffer = new_buffer
        self.p /= 2
    
    def process_stream(self, stream: List[int]) -> None:
        """Process an entire stream."""
        for element in stream:
            self.process(element)
    
    def estimate(self) -> float:
        """Return the estimated number of distinct elements."""
        if self.p > 0:
            return len(self.buffer) / self.p
        return 0
    
    def get_state(self) -> Dict:
        """Return the current state of the algorithm."""
        return {
            'buffer_size': len(self.buffer),
            'p': self.p,
            'estimate': self.estimate(),
            'subsample_count': self.subsample_count,
            'total_processed': self.total_processed
        }

In [ ]:
# Test CVM on a simple stream
stream = generate_uniform_stream(100, 5000)
true_f0 = len(set(stream))

cvm = CVMDistinctElements(buffer_size=50)
cvm.process_stream(stream)

print(f"Stream length: {len(stream)}")
print(f"True F₀: {true_f0}")
print(f"CVM estimate: {cvm.estimate():.1f}")
print(f"Relative error: {100 * abs(cvm.estimate() - true_f0) / true_f0:.1f}%")
print(f"\nAlgorithm state:")
for k, v in cvm.get_state().items():
    print(f"  {k}: {v}")

## Part 4: Analyzing the Algorithm

Let's run multiple experiments to understand the algorithm's behavior.

In [ ]:
def run_cvm_experiments(
    true_f0: int,
    stream_length: int,
    buffer_size: int,
    num_runs: int = 100,
    stream_type: str = 'uniform'
) -> List[float]:
    """Run multiple CVM experiments and return estimates."""
    estimates = []
    
    for _ in range(num_runs):
        if stream_type == 'uniform':
            stream = generate_uniform_stream(true_f0, stream_length)
        elif stream_type == 'zipf':
            stream = generate_zipf_stream(true_f0, stream_length)
        elif stream_type == 'adversarial':
            stream = generate_adversarial_stream(true_f0, stream_length)
        else:
            raise ValueError(f"Unknown stream type: {stream_type}")
        
        cvm = CVMDistinctElements(buffer_size=buffer_size)
        cvm.process_stream(stream)
        estimates.append(cvm.estimate())
    
    return estimates

In [ ]:
# Run experiments with different buffer sizes
true_f0 = 500
stream_length = 10000
buffer_sizes = [10, 20, 50, 100, 200]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Distribution of estimates for each buffer size
ax = axes[0]
positions = range(len(buffer_sizes))
all_estimates = []

for bs in buffer_sizes:
    estimates = run_cvm_experiments(true_f0, stream_length, bs, num_runs=100)
    all_estimates.append(estimates)

bp = ax.boxplot(all_estimates, positions=positions, widths=0.6, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#a8d4f0')
    
ax.axhline(y=true_f0, color='green', linestyle='--', linewidth=2, label=f'True F₀ = {true_f0}')
ax.axhline(y=true_f0 * 1.1, color='green', linestyle=':', alpha=0.5)
ax.axhline(y=true_f0 * 0.9, color='green', linestyle=':', alpha=0.5)
ax.fill_between([-0.5, len(buffer_sizes)-0.5], true_f0*0.9, true_f0*1.1, alpha=0.1, color='green')

ax.set_xticks(positions)
ax.set_xticklabels([str(bs) for bs in buffer_sizes])
ax.set_xlabel('Buffer Size')
ax.set_ylabel('Estimate')
ax.set_title(f'CVM Estimates vs Buffer Size\n(True F₀ = {true_f0}, stream length = {stream_length})')
ax.legend()

# Right plot: Error vs buffer size
ax = axes[1]
mean_errors = []
std_errors = []
for estimates in all_estimates:
    relative_errors = [abs(e - true_f0) / true_f0 for e in estimates]
    mean_errors.append(np.mean(relative_errors) * 100)
    std_errors.append(np.std(relative_errors) * 100)

ax.errorbar(buffer_sizes, mean_errors, yerr=std_errors, marker='o', capsize=5, 
            color='#3498db', linewidth=2, markersize=8)
ax.set_xlabel('Buffer Size')
ax.set_ylabel('Mean Relative Error (%)')
ax.set_title('Error Decreases with Buffer Size')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare performance on different stream types
true_f0 = 200
stream_length = 5000
buffer_size = 50
stream_types = ['uniform', 'zipf', 'adversarial']

fig, ax = plt.subplots(figsize=(10, 6))

all_estimates = {}
for st in stream_types:
    estimates = run_cvm_experiments(true_f0, stream_length, buffer_size, 
                                    num_runs=100, stream_type=st)
    all_estimates[st] = estimates

colors = {'uniform': '#3498db', 'zipf': '#e74c3c', 'adversarial': '#9b59b6'}
positions = {st: i for i, st in enumerate(stream_types)}

for st, estimates in all_estimates.items():
    jitter = np.random.normal(0, 0.08, len(estimates))
    ax.scatter([positions[st] + j for j in jitter], estimates, 
               alpha=0.4, s=30, c=colors[st], label=st.capitalize())

ax.axhline(y=true_f0, color='green', linestyle='--', linewidth=2, label=f'True F₀ = {true_f0}')
ax.fill_between([-0.5, len(stream_types)-0.5], true_f0*0.9, true_f0*1.1, 
                alpha=0.1, color='green', label='±10% band')

ax.set_xticks(range(len(stream_types)))
ax.set_xticklabels([st.capitalize() for st in stream_types])
ax.set_xlabel('Stream Type')
ax.set_ylabel('Estimate')
ax.set_title(f'CVM Performance on Different Stream Types\n(buffer = {buffer_size}, True F₀ = {true_f0})')
ax.legend()

plt.tight_layout()
plt.show()

# Print statistics
print("\nStatistics:")
for st, estimates in all_estimates.items():
    in_band = sum(1 for e in estimates if 0.9*true_f0 <= e <= 1.1*true_f0)
    print(f"  {st.capitalize():12s}: Mean={np.mean(estimates):.1f}, "
          f"Std={np.std(estimates):.1f}, In ±10% band: {in_band}%")

## Part 5: Step-by-Step Visualization

Let's watch the algorithm evolve over time.

In [ ]:
class CVMWithHistory(CVMDistinctElements):
    """CVM with history tracking for visualization."""
    
    def __init__(self, buffer_size: int):
        super().__init__(buffer_size)
        self.history = {
            'time': [],
            'buffer_size': [],
            'p': [],
            'estimate': [],
            'subsample_events': []
        }
    
    def process(self, element: int) -> None:
        old_p = self.p
        super().process(element)
        
        # Record history
        self.history['time'].append(self.total_processed)
        self.history['buffer_size'].append(len(self.buffer))
        self.history['p'].append(self.p)
        self.history['estimate'].append(self.estimate())
        
        # Track subsample events
        if self.p < old_p:
            self.history['subsample_events'].append(self.total_processed)

In [ ]:
# Generate a stream and track the algorithm's evolution
true_f0 = 150
stream_length = 3000
buffer_size = 30

stream = generate_uniform_stream(true_f0, stream_length)
actual_f0 = len(set(stream))

cvm = CVMWithHistory(buffer_size=buffer_size)
cvm.process_stream(stream)

# Plot the evolution
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Plot 1: Buffer size over time
ax = axes[0]
ax.plot(cvm.history['time'], cvm.history['buffer_size'], color='#3498db', linewidth=1)
ax.axhline(y=buffer_size, color='red', linestyle='--', label=f'Max buffer = {buffer_size}')
for t in cvm.history['subsample_events']:
    ax.axvline(x=t, color='orange', alpha=0.5, linestyle=':')
ax.set_ylabel('Buffer Size |X|')
ax.set_title('CVM Algorithm Evolution')
ax.legend()

# Plot 2: Probability p over time
ax = axes[1]
ax.plot(cvm.history['time'], cvm.history['p'], color='#9b59b6', linewidth=1.5)
ax.set_ylabel('Probability p')
ax.set_yscale('log')
for t in cvm.history['subsample_events']:
    ax.axvline(x=t, color='orange', alpha=0.5, linestyle=':', label='Subsample' if t == cvm.history['subsample_events'][0] else '')
ax.legend()

# Plot 3: Estimate over time
ax = axes[2]
ax.plot(cvm.history['time'], cvm.history['estimate'], color='#2ecc71', linewidth=1, alpha=0.7)
ax.axhline(y=actual_f0, color='green', linestyle='--', linewidth=2, label=f'True F₀ = {actual_f0}')
ax.fill_between(cvm.history['time'], actual_f0*0.9, actual_f0*1.1, alpha=0.1, color='green')
ax.set_ylabel('Estimate |X|/p')
ax.set_xlabel('Stream Position')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nFinal estimate: {cvm.estimate():.1f}")
print(f"True F₀: {actual_f0}")
print(f"Number of subsample events: {len(cvm.history['subsample_events'])}")
print(f"Final p: {cvm.p}")

## Part 6: Theoretical Analysis - Verifying Chernoff Bounds

Let's verify that the algorithm's accuracy matches the theoretical predictions.

In [ ]:
def theoretical_error_bound(buffer_size: int, delta: float = 0.1) -> float:
    """
    Compute the theoretical ε such that the algorithm gives (1±ε) approximation
    with probability ≥ 1-δ.
    
    From Chernoff: Pr[|X - μ| > εμ] < 2exp(-ε²μ/3)
    Setting μ ≈ buffer_size/2 and failure prob = δ:
    ε ≈ sqrt(6 * ln(2/δ) / buffer_size)
    """
    import math
    return math.sqrt(6 * math.log(2 / delta) / buffer_size)


# Compare theoretical vs empirical error
buffer_sizes = [10, 20, 30, 50, 75, 100, 150, 200]
true_f0 = 500
stream_length = 10000
num_runs = 200

theoretical_eps = [theoretical_error_bound(bs) for bs in buffer_sizes]
empirical_eps = []

for bs in buffer_sizes:
    estimates = run_cvm_experiments(true_f0, stream_length, bs, num_runs=num_runs)
    # Compute 90th percentile of relative errors
    rel_errors = [abs(e - true_f0) / true_f0 for e in estimates]
    empirical_eps.append(np.percentile(rel_errors, 90))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(buffer_sizes, theoretical_eps, 'o-', color='#e74c3c', linewidth=2, 
        markersize=8, label='Theoretical ε (Chernoff bound)')
ax.plot(buffer_sizes, empirical_eps, 's-', color='#3498db', linewidth=2, 
        markersize=8, label='Empirical ε (90th percentile)')

ax.set_xlabel('Buffer Size')
ax.set_ylabel('ε (Relative Error)')
ax.set_title('Theoretical vs Empirical Error Bounds')
ax.legend()
ax.grid(True, alpha=0.3)

# Add 1/√s reference line
ref_line = [1.5 / np.sqrt(bs) for bs in buffer_sizes]
ax.plot(buffer_sizes, ref_line, '--', color='gray', alpha=0.5, label='O(1/√buffer_size)')
ax.legend()

plt.tight_layout()
plt.show()

## Part 7: Comparison with Exact Counting

Let's compare the memory and time trade-offs.

In [ ]:
def exact_distinct_count(stream: List[int]) -> int:
    """Exact counting using a hash set."""
    return len(set(stream))


def memory_usage_comparison(true_f0: int, stream_length: int, buffer_size: int):
    """Compare memory usage of exact vs approximate methods."""
    # Exact: stores all distinct elements
    exact_memory = true_f0  # In terms of element count
    
    # CVM: stores at most buffer_size elements plus O(1) for p
    cvm_memory = buffer_size + 1
    
    return {
        'exact': exact_memory,
        'cvm': cvm_memory,
        'savings': (exact_memory - cvm_memory) / exact_memory * 100
    }


# Compare for different F₀ values
f0_values = [100, 500, 1000, 5000, 10000, 50000]
buffer_size = 100

print(f"Memory comparison (buffer size = {buffer_size}):\n")
print(f"{'True F₀':>10} | {'Exact Memory':>15} | {'CVM Memory':>12} | {'Savings':>10}")
print("-" * 55)

for f0 in f0_values:
    mem = memory_usage_comparison(f0, f0 * 10, buffer_size)
    print(f"{f0:>10} | {mem['exact']:>15} | {mem['cvm']:>12} | {mem['savings']:>9.1f}%")

## Part 8: Your Turn! Exercises

Try these exercises to deepen your understanding:

In [ ]:
# Exercise 1: Implement the Flajolet-Martin algorithm (hash-based)
# Hint: Use a hash function h and track max trailing zeros in h(x) for all x

def flajolet_martin(stream: List[int]) -> float:
    """
    TODO: Implement Flajolet-Martin algorithm.
    
    Idea:
    1. Use a hash function that maps elements to integers
    2. For each element, compute the number of trailing zeros in its hash
    3. Track the maximum number of trailing zeros seen (call it R)
    4. Estimate F₀ ≈ 2^R
    """
    pass  # Your implementation here

In [ ]:
# Exercise 2: Modify CVM to support element deletions
# (This is called the "turnstile model")

class CVMTurnstile(CVMDistinctElements):
    """
    TODO: Modify CVM to handle both insertions (+1) and deletions (-1).
    
    Challenge: The standard CVM doesn't work for deletions. 
    Can you think of how to modify it, or prove it's impossible?
    """
    pass  # Your implementation here

In [ ]:
# Exercise 3: Implement a sliding window version
# Only count distinct elements in the last W elements

class CVMSlidingWindow:
    """
    TODO: Estimate F₀ over a sliding window of the last W elements.
    
    This is significantly harder! Think about:
    - How do you "forget" old elements?
    - How do you maintain the invariant when elements expire?
    """
    pass  # Your implementation here

## References

1. **CVM Algorithm**: Chakraborty, S., Vinodchandran, N. V., & Meel, K. S. (2022). *Distinct Elements in Streams: An Algorithm for the (Text) Book*. European Symposium on Algorithms (ESA). [arXiv:2301.10191](https://arxiv.org/abs/2301.10191)

2. **Misra-Gries**: Misra, J., & Gries, D. (1982). *Finding repeated elements*. Science of Computer Programming.

3. **Flajolet-Martin**: Flajolet, P., & Martin, G. N. (1985). *Probabilistic counting algorithms for data base applications*. Journal of Computer and System Sciences.

4. **HyperLogLog**: Flajolet, P., Fusy, É., Gandouet, O., & Meunier, F. (2007). *HyperLogLog: the analysis of a near-optimal cardinality estimation algorithm*. DMTCS Proceedings.

5. **Knuth's Notes**: Knuth, D. E. (2024). *The CVM Algorithm for Estimating Distinct Elements in Streams*. [Stanford](https://www-cs-faculty.stanford.edu/~knuth/papers/cvm-note.pdf)

6. **Survey**: Muthukrishnan, S. (2005). *Data Streams: Algorithms and Applications*. NOW Publishers.